# Delta Lake: スキーマ強制とスキーマ変更（V2 · お客様説明用）

Delta テーブルの「スキーマ強制」と「スキーマ変更」に関する挙動を、必要最小限のシナリオで確認します。<br>
**セルを 1 つずつ実行してください。** エラーになるセルはタイトルに「→ エラー」と明記しています（意図した挙動です）。

お客様への説明はメイン 3 シナリオで完結します。<br>
「他にもパターンがあります」の紹介として、末尾にオプショナルなシナリオをまとめています。

## 全体マップ

### メイン（お客様説明用）

| # | 検証項目 | 想定結果 | 事前の有効化 |
|---|---|---|---|
| 1 | 存在しない列を入れる | エラー（スキーマ強制） | 不要 |
| 2 | 列を追加する（`ADD COLUMNS`）| 成功 | 不要 |
| 3 | MERGE で新列を含めて差分反映 | 成功 | 構文で明示（テーブル側の事前設定は不要）|

### オプショナル（他のパターンの紹介）

| # | 検証項目 | 想定結果 | 事前の有効化 |
|---|---|---|---|
| 4 | 型が違う値を入れる | エラー | 不要 |
| 5 | 列を削除する（`DROP COLUMN`）| エラー → 有効化後に成功 | `delta.columnMapping.mode = 'name'` |
| 6 | 列名を変更する（`RENAME COLUMN`）| 成功 | 5 と同じプロパティ |
| 7 | 型を広げる（`ALTER COLUMN ... TYPE`）| 成功 | `delta.enableTypeWidening = true` |
| 8 | INSERT で新列を追加したい | （2 と同じ）| 事前に `ADD COLUMNS` |

## 準備

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS konomi_demo_catalog.schema_evolution_demo;
USE konomi_demo_catalog.schema_evolution_demo;

DROP TABLE IF EXISTS sales;

CREATE TABLE sales (
  sale_id     BIGINT,
  store_id    STRING,
  product_id  STRING,
  amount      DECIMAL(10, 2),
  sale_date   DATE
);

INSERT INTO sales VALUES
  (1, 'S001', 'P100', 1200.00, DATE'2026-07-01'),
  (2, 'S001', 'P101',  850.00, DATE'2026-07-01'),
  (3, 'S002', 'P100', 1500.00, DATE'2026-07-02');

SELECT * FROM sales ORDER BY sale_id;

# メイン

お客様への説明はここまで（1 → 2 → 3）で完結します。

## 1. 存在しない列を入れる → エラー

Delta の **スキーマ強制**により、テーブルに存在しない列を含む INSERT は自動的に弾かれます。<br>
想定外のデータ混入からテーブルを守る仕組みです。

In [0]:
%sql
INSERT INTO sales (sale_id, store_id, product_id, amount, sale_date, campaign_id)
VALUES (4, 'S002', 'P102', 980.00, DATE'2026-07-03', 'CP001');

## 2. 列を追加する（`ALTER TABLE ADD COLUMNS`）→ 成功

一番よくあるスキーマ変更（カラム追加）は、`ALTER TABLE ADD COLUMNS` で追加できます。<br>
プロトコルの変更や事前準備は不要で、既存データに影響しません。

参考：[ALTER TABLE](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-alter-table)

In [0]:
%sql
ALTER TABLE sales ADD COLUMNS (campaign_id STRING);

INSERT INTO sales (sale_id, store_id, product_id, amount, sale_date, campaign_id)
VALUES (4, 'S002', 'P102', 980.00, DATE'2026-07-03', 'CP001');

SELECT * FROM sales ORDER BY sale_id;

## 3. MERGE で新列を含めて差分反映 → 成功

実運用の ETL では、ソース側に新しい列が追加されるケースがよくあります。<br>
`MERGE WITH SCHEMA EVOLUTION` を使うと、テーブル側の事前準備なしで **差分反映とスキーマ拡張を同時に**行えます。

参考：[MERGE INTO - Automatic schema evolution](https://docs.databricks.com/aws/en/sql/language-manual/delta-merge-into#automatic-schema-evolution)

In [0]:
%sql
-- 差分ソース（新列 discount_rate を含む）
CREATE OR REPLACE TEMP VIEW sales_updates AS
SELECT
  sale_id,
  store_id,
  product_id,
  CAST(amount AS DECIMAL(10,2))       AS amount,
  sale_date,
  CAST(discount_rate AS DECIMAL(5,4)) AS discount_rate
FROM VALUES
  (2, 'S001', 'P101',  800.00, DATE'2026-07-01', 0.05),
  (6, 'S004', 'P104', 3300.00, DATE'2026-07-05', 0.15)
AS t(sale_id, store_id, product_id, amount, sale_date, discount_rate);

MERGE WITH SCHEMA EVOLUTION INTO sales AS target
USING sales_updates AS source
ON target.sale_id = source.sale_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT * FROM sales ORDER BY sale_id;

## メインまとめ

| # | 操作 | 挙動 |
|---|---|---|
| 1 | 想定外の列を弾く | スキーマ強制でエラー |
| 2 | 列追加 | `ADD COLUMNS` で成功、事前設定不要 |
| 3 | MERGE で新列を含めて差分反映 | `MERGE WITH SCHEMA EVOLUTION` で成功 |

# オプショナル

「他にもこんなパターンがあります」の紹介として、追加のシナリオを掲載します。<br>
お客様への説明では時間があれば触れる程度で構いません。

## 4. 型が違う値を入れる → エラー

列名は合っていても、型が違う値は弾かれます（スキーマ強制の一部）。

In [0]:
%sql
INSERT INTO sales (sale_id, store_id, product_id, amount, sale_date)
VALUES (5, 'S002', 'P102', 'not_a_number', DATE'2026-07-03');

## 5. 列を削除する（`ALTER TABLE DROP COLUMN`）→ 未設定でエラー、有効化後に成功

参考：[Delta Lake column mapping](https://docs.databricks.com/aws/en/delta/column-mapping)

<div style="border-left: 4px solid #D32F2F; background: #FFEBEE; padding: 12px 16px; border-radius: 4px;">
<b>本番テーブルで有効化する前に必ず確認してください</b>
<ul>
  <li>一度 <code>name</code> にすると原則として元に戻せません（プロトコルバージョンがアップグレードされるため）</li>
  <li>そのテーブルを読む側（外部 Spark、古い Delta Reader、Parquet 直接参照など）が column mapping に対応しているか事前確認が必要</li>
  <li>非対応の reader からは読めなくなる可能性があります</li>
</ul>
</div>

In [0]:
%sql
-- 有効化なしで DROP COLUMN → エラー
ALTER TABLE sales DROP COLUMN campaign_id;

In [0]:
%sql
-- プロパティを有効化してから再実行 → 成功
ALTER TABLE sales SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

ALTER TABLE sales DROP COLUMN campaign_id;

DESCRIBE TABLE sales;

## 6. 列名を変更する（`ALTER TABLE RENAME COLUMN`）→ 成功

5 で `columnMapping.mode = 'name'` を設定済みのため、そのまま使えます。

In [0]:
%sql
ALTER TABLE sales RENAME COLUMN store_id TO shop_id;

SELECT * FROM sales ORDER BY sale_id;

## 7. 型を広げる（`ALTER TABLE ALTER COLUMN ... TYPE`）→ 成功

参考：[Type widening](https://docs.databricks.com/aws/en/delta/type-widening)

<div style="border-left: 4px solid #D32F2F; background: #FFEBEE; padding: 12px 16px; border-radius: 4px;">
<b>本番テーブルで有効化する前に必ず確認してください</b>
<ul>
  <li>プロトコルバージョンがアップグレードされ、原則として元に戻せません</li>
  <li>Databricks Runtime のバージョン要件を満たしていないと、そのテーブルを読み書きできなくなる可能性があります</li>
  <li>拡張方向のみサポート（INT → BIGINT / DECIMAL の桁数拡大など）</li>
</ul>
</div>

In [0]:
%sql
-- Before: INT で列を追加
ALTER TABLE sales ADD COLUMNS (store_visit_count INT);
DESCRIBE TABLE sales;

In [0]:
%sql
-- After: Type Widening を有効化して INT → BIGINT に拡張
ALTER TABLE sales SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true');
ALTER TABLE sales ALTER COLUMN store_visit_count TYPE BIGINT;
DESCRIBE TABLE sales;

## 8. INSERT で新列を追加したい場合

2 と同じで、先に `ALTER TABLE ADD COLUMNS` で列を追加してから通常の INSERT を実行します。<br>
INSERT 側から動的に列を増やす方法は Delta にはありません。ETL で列拡張が発生する場合は 3（MERGE）を推奨。

## オプショナルまとめ

| # | 操作 | 想定結果 | 事前の有効化 |
|---|---|---|---|
| 4 | 型不一致を弾く | エラー | 不要 |
| 5 | 列削除 | エラー → 有効化後に成功 | `delta.columnMapping.mode = 'name'` |
| 6 | 列名変更 | 成功 | 5 と同じプロパティ |
| 7 | 型を広げる | 成功 | `delta.enableTypeWidening = true` |
| 8 | INSERT で新列 | 成功 | 事前に `ADD COLUMNS` |